In [1]:
import os
import cv2
import numpy as np
import rasterio
from rasterio.windows import Window
from glob import glob

In [2]:
# ==============================
# CONFIGURATION
# ==============================

PATCH_SIZE = 192
SCALE = 2
DATASET_ROOT = r"A:\Main Project\Satellite_SR\dataset"

SAFE_DIRECTORIES = [
    r"A:\Main Project\Satellite_SR\data\City1_T43PGQ",
    r"A:\Main Project\Satellite_SR\data\City2_T44PLV"
]

# Create dataset folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(DATASET_ROOT, split, "HR"), exist_ok=True)
    os.makedirs(os.path.join(DATASET_ROOT, split, "LR"), exist_ok=True)

In [7]:
# ==============================
# FUNCTION: Get SAFE files
# ==============================

def get_safe_files():
    safe_files = []
    for base in SAFE_DIRECTORIES:
        for folder in os.listdir(base):
            full_path = os.path.join(base, folder)
            if os.path.isdir(full_path):
                # Look inside this folder for .SAFE
                safe_inside = glob(os.path.join(full_path, "*.SAFE"))
                safe_files.extend(safe_inside)
    return safe_files



In [8]:
safe_files = get_safe_files()
print("SAFE files found:", len(safe_files))


SAFE files found: 4


In [6]:
for base in SAFE_DIRECTORIES:
    print("Checking:", base)
    print("Folders inside:", os.listdir(base))


Checking: A:\Main Project\Satellite_SR\data\City1_T43PGQ
Folders inside: ['S2B_MSIL2A_20251213T051119_N0511_R019_T43PGQ_20251213T071206', 'S2B_MSIL2A_20251223T051129_N0511_R019_T43PGQ_20251223T084635']
Checking: A:\Main Project\Satellite_SR\data\City2_T44PLV
Folders inside: ['S2B_MSIL2A_20250325T045659_N0511_R119_T44PLV_20250325T074633', 'S2B_MSIL2A_20250504T045659_N0511_R119_T44PLV_20250504T072305']


In [9]:
# ==============================
# START DATASET GENERATION
# ==============================

safe_files = get_safe_files()
patch_counter = 0

print("SAFE files found:", len(safe_files))

for safe_idx, safe_path in enumerate(safe_files):
    print(f"\nProcessing {safe_idx+1}/{len(safe_files)}:", os.path.basename(safe_path))

    # Find R10m folder
    granules = glob(os.path.join(safe_path, "GRANULE", "*"))
    granule_path = None
    for g in granules:
        if os.path.exists(os.path.join(g, "IMG_DATA", "R10m")):
            granule_path = g
            break

    r10m_path = os.path.join(granule_path, "IMG_DATA", "R10m")

    b02_path = glob(os.path.join(r10m_path, "*_B02_10m.jp2"))[0]
    b03_path = glob(os.path.join(r10m_path, "*_B03_10m.jp2"))[0]
    b04_path = glob(os.path.join(r10m_path, "*_B04_10m.jp2"))[0]

    with rasterio.open(b02_path) as src_b02, \
         rasterio.open(b03_path) as src_b03, \
         rasterio.open(b04_path) as src_b04:

        height, width = src_b02.height, src_b02.width

        for row in range(0, height - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, width - PATCH_SIZE + 1, PATCH_SIZE):

                window = Window(col, row, PATCH_SIZE, PATCH_SIZE)

                try:
                    B02 = src_b02.read(1, window=window)
                    B03 = src_b03.read(1, window=window)
                    B04 = src_b04.read(1, window=window)
                except:
                    continue  # Skip problematic tiles

                # Skip empty patches
                if B04.max() == 0:
                    continue

                # Stack RGB (float32)
                hr = np.stack([B04, B03, B02], axis=-1).astype("float32")

                # Normalize per patch
                hr = hr / hr.max()

                # Create LR (192 → 96)
                lr = cv2.resize(
                    hr,
                    (PATCH_SIZE // SCALE, PATCH_SIZE // SCALE),
                    interpolation=cv2.INTER_AREA
                )

                # Random split
                r = np.random.rand()
                if r < 0.7:
                    split = "train"
                elif r < 0.85:
                    split = "val"
                else:
                    split = "test"

                hr_path = os.path.join(DATASET_ROOT, split, "HR", f"img_{patch_counter:06d}.png")
                lr_path = os.path.join(DATASET_ROOT, split, "LR", f"img_{patch_counter:06d}.png")

                cv2.imwrite(hr_path, (hr * 255).astype(np.uint8))
                cv2.imwrite(lr_path, (lr * 255).astype(np.uint8))

                patch_counter += 1

    print("  Patches saved so far:", patch_counter)

print("\n✅ DATASET REGENERATION COMPLETED")
print("Total patches saved:", patch_counter)

SAFE files found: 4

Processing 1/4: S2B_MSIL2A_20251213T051119_N0511_R019_T43PGQ_20251213T071206.SAFE
  Patches saved so far: 3218

Processing 2/4: S2B_MSIL2A_20251223T051129_N0511_R019_T43PGQ_20251223T084635.SAFE
  Patches saved so far: 6438

Processing 3/4: S2B_MSIL2A_20250325T045659_N0511_R119_T44PLV_20250325T074633.SAFE
  Patches saved so far: 9182

Processing 4/4: S2B_MSIL2A_20250504T045659_N0511_R119_T44PLV_20250504T072305.SAFE
  Patches saved so far: 11937

✅ DATASET REGENERATION COMPLETED
Total patches saved: 11937


In [11]:
import cv2

sample_hr = cv2.imread(
    r"A:\Main Project\Satellite_SR\dataset\train\HR\img_000000.png"
)

sample_lr = cv2.imread(
    r"A:\Main Project\Satellite_SR\dataset\train\LR\img_000000.png"
)

print("HR shape:", sample_hr.shape)
print("LR shape:", sample_lr.shape)



HR shape: (192, 192, 3)
LR shape: (96, 96, 3)
